

### Funcionalidades:
-  Alertas de voz: *"Persona al centro, muy cerca"*
-  Zonas espaciales: izquierda / centro / derecha
-  3 niveles de peligro con prioridad de voz
-  **Modo dual**: usa tu modelo entrenado O el modelo COCO base
-  Teclas de control en tiempo real

##  Configuración general
Aquí ajustas rutas, umbrales y preferencias de voz.

In [5]:
import os

# ═══════════════════════════════════════════════════════════
#  SECCIÓN A — MODELO A USAR
#  Cambia USE_MODELO_PROPIO para alternar entre modos
# ═══════════════════════════════════════════════════════════

USE_MODELO_PROPIO = True   # True = tu modelo entrenado | False = COCO base

# Ruta a tu modelo entrenado (best.pt)
# Ajusta esta ruta si tu proyecto está en otro lugar
RUTA_MODELO_PROPIO = r"runs\detect\runs\detect\deteccion_cercana\weights\best.pt"

# Modelo base de YOLOv8 (se descarga automáticamente si no existe)
RUTA_MODELO_COCO  = "yolov8n.pt"

# Clases de TU modelo entrenado (data.yaml)
CLASES_MODELO_PROPIO = {
    0: "carro",
    1: "moto",
    2: "persona",
    3: "poste",
    4: "señal de tránsito",
}

# Clases relevantes del modelo COCO base (solo las útiles para navegación)
CLASES_MODELO_COCO = {
    0:  "persona",
    1:  "bicicleta",
    2:  "auto",
    3:  "moto",
    9:  "semáforo",
    11: "señal de pare",
    13: "banco",
    56: "silla",
}

# ═══════════════════════════════════════════════════════════
#  SECCIÓN C — DETECCIÓN
# ═══════════════════════════════════════════════════════════

CONFIANZA_MIN = 0.45      # Umbral de confianza mínimo
CAMARA_INDEX  = 0         # 0 = cámara principal, 1 = externa

# Umbrales de área para niveles de alerta (ratio bbox/frame)
UMBRAL_PELIGRO  = 0.20    # >20% del frame → peligro (muy cerca)
UMBRAL_ATENCION = 0.07    # >7%  del frame → atención (distancia media)

# ═══════════════════════════════════════════════════════════
#  SECCIÓN D — AUDIO
# ═══════════════════════════════════════════════════════════

# Intervalo mínimo entre alertas del MISMO objeto (segundos)
# Evita que la voz repita sin parar el mismo mensaje
COOLDOWN_OBJETO = 3.5

# Intervalo mínimo entre cualquier alerta de peligro (segundos)
COOLDOWN_PELIGRO = 1.5

# Velocidad de voz (palabras por minuto). 150=normal, 120=lento, 180=rápido
VELOCIDAD_VOZ = 145

# Volumen de voz (0.0 a 1.0)
VOLUMEN_VOZ = 1.0

# ═══════════════════════════════════════════════════════════
#  SECCIÓN E — VISUALIZACIÓN
# ═══════════════════════════════════════════════════════════

COLOR_SEGURO   = (0, 200, 0)      # Verde
COLOR_ATENCION = (0, 165, 255)    # Naranja
COLOR_PELIGRO  = (0, 0, 220)      # Rojo

# ── Selección automática de modelo y clases ──
if USE_MODELO_PROPIO:
    MODELO_PATH  = RUTA_MODELO_PROPIO
    CLASES_ACTIVAS = CLASES_MODELO_PROPIO
    FILTRAR_CLASES = False   # El modelo propio ya detecta solo lo entrenado
    print(f" Modo: MODELO PROPIO")
    print(f"   Ruta: {MODELO_PATH}")
    if not os.path.exists(MODELO_PATH):
        print(f"  ADVERTENCIA: No se encontró best.pt en {MODELO_PATH}")
        print("    Verifica la ruta o cambia USE_MODELO_PROPIO = False")
else:
    MODELO_PATH  = RUTA_MODELO_COCO
    CLASES_ACTIVAS = CLASES_MODELO_COCO
    FILTRAR_CLASES = True    # COCO detecta 80 clases, filtramos las útiles
    print(f" Modo: MODELO COCO BASE")
    print(f"   Modelo: {MODELO_PATH}")

print(f"\n📋 Clases activas ({len(CLASES_ACTIVAS)}):")
for id_, nombre in CLASES_ACTIVAS.items():
    print(f"   [{id_}] {nombre}")

 Modo: MODELO PROPIO
   Ruta: runs\detect\runs\detect\deteccion_cercana\weights\best.pt

📋 Clases activas (5):
   [0] carro
   [1] moto
   [2] persona
   [3] poste
   [4] señal de tránsito


In [6]:
# ═══ DIAGNÓSTICO DE AUDIO ═══
import pyttsx3
import threading
import queue

print("=== Diagnóstico TTS ===")

# Paso 1: motor directo (sin hilos)
print("\n1. Probando motor directo...")
try:
    motor_test = pyttsx3.init()
    voces = motor_test.getProperty('voices')
    print(f"   Voces disponibles: {len(voces)}")
    for i, v in enumerate(voces):
        marca = "👉" if ("spanish" in v.name.lower() or "es_" in v.id.lower()) else "  "
        print(f"   {marca} [{i}] {v.name}")
    
    motor_test.say("prueba uno dos tres")
    motor_test.runAndWait()
    print("   ¿Escuchaste 'prueba uno dos tres'? → si sí, el motor funciona")
except Exception as e:
    print(f"   ❌ Error: {e}")

=== Diagnóstico TTS ===

1. Probando motor directo...
   Voces disponibles: 3
   👉 [0] Microsoft Sabina Desktop - Spanish (Mexico)
      [1] Microsoft Zira Desktop - English (United States)
      [2] Microsoft David Desktop - English (United States)
   ¿Escuchaste 'prueba uno dos tres'? → si sí, el motor funciona


## Celda 3 — Motor de voz (TTS)
Inicializa pyttsx3 con español y configura un hilo dedicado para el audio.  
El audio corre en un **hilo separado** para no bloquear la cámara.

In [15]:
import subprocess
import threading
import queue
import time

cola_voz = queue.Queue(maxsize=3)
_voz_activa = threading.Event()
_voz_activa.set()
_hilo_voz = None


def _worker_voz():
    """Hilo que saca mensajes de la cola y los habla con PowerShell."""
    print("[TTS] Hilo de audio iniciado ✅")
    while _voz_activa.is_set():
        try:
            mensaje = cola_voz.get(timeout=0.5)
            if mensaje is None:
                break
            # PowerShell habla directamente con SAPI5 de Windows
            # No depende de pyttsx3 ni de drivers adicionales
            cmd = (
                f"Add-Type -AssemblyName System.Speech; "
                f"$s = New-Object System.Speech.Synthesis.SpeechSynthesizer; "
                f"$s.Rate = 1; "
                f"$s.Volume = 100; "
                f'$s.SelectVoiceByHints([System.Speech.Synthesis.VoiceGender]::NotSet, '
                f'[System.Speech.Synthesis.VoiceAge]::NotSet, 0, '
                f'"es-CO"); '   # ← español Colombia, cámbialo a es-MX o es-ES si no funciona
                f'$s.Speak("{mensaje}")'
            )
            subprocess.run(
                ["powershell", "-Command", cmd],
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL
            )
            cola_voz.task_done()
        except queue.Empty:
            continue
        except Exception as e:
            print(f"[TTS] Error: {e}")
    print("[TTS] Hilo de audio cerrado.")


def iniciar_voz():
    global _hilo_voz
    _voz_activa.set()
    _hilo_voz = threading.Thread(target=_worker_voz, daemon=True)
    _hilo_voz.start()


def detener_voz():
    _voz_activa.clear()
    try:
        cola_voz.put_nowait(None)
    except queue.Full:
        pass
    if _hilo_voz and _hilo_voz.is_alive():
        _hilo_voz.join(timeout=2)


def hablar(mensaje, prioridad=False):
    if not _voz_activa.is_set():
        return
    if prioridad:
        while not cola_voz.empty():
            try:
                cola_voz.get_nowait()
            except queue.Empty:
                break
    try:
        cola_voz.put_nowait(mensaje)
    except queue.Full:
        pass


# ── Test ──
iniciar_voz()
time.sleep(0.3)
hablar("Sistema de visión asistida iniciado")
print("[INFO] Motor de voz con PowerShell listo.")
print("[INFO] Deberías escuchar el mensaje en 1-2 segundos.")

[TTS] Hilo de audio iniciado ✅
[INFO] Motor de voz con PowerShell listo.
[INFO] Deberías escuchar el mensaje en 1-2 segundos.


## Celda 4 — Motor de prioridad de alertas
Decide **cuándo y qué hablar** para evitar saturar al usuario con audio constante.

In [8]:
import time
from collections import defaultdict

# Registro de última vez que se habló de cada objeto
_ultimo_aviso = defaultdict(float)   # clave: "clase_zona" → timestamp
_ultimo_peligro = 0.0                 # timestamp del último aviso de peligro


def construir_mensaje(nombre_clase, zona, nivel):
    """
    Construye el mensaje de voz según la clase, zona y nivel de peligro.
    Ejemplos:
      "Persona al centro, muy cerca"
      "Carro a la izquierda, acercándose"
      "Moto a la derecha"
    """
    # Preposición de zona
    prep_zona = {
        "izquierda": "a la izquierda",
        "centro":    "al centro",
        "derecha":   "a la derecha",
    }.get(zona, f"en {zona}")

    # Complemento de proximidad
    desc_nivel = {
        "peligro":  ", muy cerca",
        "atencion": ", acercándose",
        "seguro":   "",
    }.get(nivel, "")

    # Capitalizar primera letra del nombre de clase
    nombre = nombre_clase.capitalize()

    return f"{nombre} {prep_zona}{desc_nivel}"


def evaluar_y_alertar(detecciones):
    """
    Recibe la lista de detecciones del frame actual y decide
    qué mensajes de voz emitir con base en prioridad y cooldowns.

    detecciones: lista de dicts con keys:
        nombre_clase, zona, nivel, confianza
    """
    global _ultimo_peligro
    ahora = time.time()

    # Ordenar: primero los de peligro, luego atención, luego seguros
    orden_nivel = {"peligro": 0, "atencion": 1, "seguro": 2}
    detecciones_ord = sorted(detecciones, key=lambda d: orden_nivel.get(d["nivel"], 3))

    mensajes_emitidos = 0

    for det in detecciones_ord:
        nombre = det["nombre_clase"]
        zona   = det["zona"]
        nivel  = det["nivel"]

        # Clave única para este objeto en esta zona
        clave = f"{nombre}_{zona}"

        # ── Reglas de cooldown ──
        if nivel == "peligro":
            # Peligro: cooldown global más corto para reaccionar rápido
            if ahora - _ultimo_peligro < COOLDOWN_PELIGRO:
                continue
            if ahora - _ultimo_aviso[clave] < COOLDOWN_OBJETO:
                continue
            _ultimo_peligro = ahora
            es_prioritario = True

        elif nivel == "atencion":
            if ahora - _ultimo_aviso[clave] < COOLDOWN_OBJETO:
                continue
            es_prioritario = False

        else:  # seguro
            # Los objetos lejanos no necesitan aviso de voz
            continue

        # ── Emitir alerta ──
        mensaje = construir_mensaje(nombre, zona, nivel)
        hablar(mensaje, prioridad=es_prioritario)
        _ultimo_aviso[clave] = ahora
        mensajes_emitidos += 1

        # Limitar a 2 alertas por frame para no solapar audio
        if mensajes_emitidos >= 2:
            break


def resetear_cooldowns():
    """Limpia todos los cooldowns. Útil para reiniciar sesión."""
    global _ultimo_peligro
    _ultimo_aviso.clear()
    _ultimo_peligro = 0.0


# ── Test del motor de prioridad ──
print("Test de mensajes:")
print(construir_mensaje("persona",  "centro",    "peligro"))
print(construir_mensaje("carro",    "izquierda", "atencion"))
print(construir_mensaje("moto",     "derecha",   "seguro"))
print(construir_mensaje("poste",    "centro",    "peligro"))
print("\n✅ Motor de prioridad configurado.")

Test de mensajes:
Persona al centro, muy cerca
Carro a la izquierda, acercándose
Moto a la derecha
Poste al centro, muy cerca

✅ Motor de prioridad configurado.


## Celda 5 — Funciones de detección y visualización
Reutiliza y extiende la lógica de `detector.py` (semana 1).

In [9]:
import cv2
import torch


def verificar_gpu():
    if torch.cuda.is_available():
        nombre = torch.cuda.get_device_name(0)
        vram   = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"[GPU] {nombre} | VRAM: {vram:.1f} GB")
        return "cuda"
    print("[ADVERTENCIA] GPU no disponible. Usando CPU.")
    return "cpu"


def calcular_zona(cx, ancho_frame):
    """Devuelve 'izquierda', 'centro' o 'derecha'."""
    tercio = ancho_frame // 3
    if cx < tercio:
        return "izquierda"
    elif cx < 2 * tercio:
        return "centro"
    return "derecha"


def calcular_nivel_alerta(bbox_area, frame_area):
    """Devuelve 'peligro', 'atencion' o 'seguro' según ratio de área."""
    ratio = bbox_area / frame_area
    if ratio > UMBRAL_PELIGRO:
        return "peligro"
    elif ratio > UMBRAL_ATENCION:
        return "atencion"
    return "seguro"


def dibujar_deteccion(frame, box, nombre_clase, confianza, nivel, zona):
    """Dibuja el bounding box con etiqueta y color según nivel."""
    x1, y1, x2, y2 = map(int, box)

    color = {
        "peligro":  COLOR_PELIGRO,
        "atencion": COLOR_ATENCION,
        "seguro":   COLOR_SEGURO,
    }.get(nivel, COLOR_SEGURO)

    grosor = 3 if nivel == "peligro" else 2
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, grosor)

    prefijo  = "[!] " if nivel == "peligro" else ""
    etiqueta = f"{prefijo}{nombre_clase} | {confianza:.0%} | {zona}"

    (tw, th), _ = cv2.getTextSize(etiqueta, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
    cv2.rectangle(frame, (x1, y1 - th - 8), (x1 + tw + 4, y1), color, -1)
    cv2.putText(frame, etiqueta, (x1 + 2, y1 - 4),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)
    return frame


def dibujar_hud(frame, fps, n_objetos, n_peligros, modo_modelo, voz_on):
    """HUD extendido con indicador de modo de modelo y estado de voz."""
    h, w = frame.shape[:2]

    # Barra superior
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (w, 50), (20, 20, 20), -1)
    cv2.addWeighted(overlay, 0.65, frame, 0.35, 0, frame)

    # FPS y objetos
    cv2.putText(frame, f"FPS: {fps:.1f}",        (10, 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)
    cv2.putText(frame, f"Obj: {n_objetos}",       (110, 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)

    # Indicador de peligro
    if n_peligros > 0:
        cv2.putText(frame, f"⚠ PELIGRO ({n_peligros})",
                    (w // 2 - 80, 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 0, 255), 2)

    # Modo de modelo (esquina superior derecha)
    txt_modo  = "PROPIO" if modo_modelo else "COCO"
    col_modo  = (0, 255, 150) if modo_modelo else (150, 200, 255)
    cv2.putText(frame, f"Modelo: {txt_modo}", (w - 170, 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, col_modo, 1)

    # Estado de voz (segunda línea HUD)
    txt_voz = "🔊 VOZ: ON " if voz_on else "🔇 VOZ: OFF"
    col_voz = (0, 255, 100) if voz_on else (80, 80, 80)
    cv2.putText(frame, txt_voz, (10, 42),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, col_voz, 1)

    # Ayuda de teclas (segunda línea derecha)
    cv2.putText(frame, "[M] modelo  [V] voz  [Q] salir",
                (w - 270, 42),
                cv2.FONT_HERSHEY_SIMPLEX, 0.42, (130, 130, 130), 1)

    # Líneas divisoras de zonas
    tercio = w // 3
    cv2.line(frame, (tercio, 50),     (tercio, h),     (80, 80, 80), 1)
    cv2.line(frame, (2*tercio, 50),   (2*tercio, h),   (80, 80, 80), 1)

    for i, nombre_zona in enumerate(["◄ Izquierda", "  Centro", "Derecha ►"]):
        x_zona = tercio * i + 10
        cv2.putText(frame, nombre_zona, (x_zona, h - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.42, (100, 100, 100), 1)

    return frame


print("✅ Funciones de detección y visualización cargadas.")

✅ Funciones de detección y visualización cargadas.


## Celda 6 — Bucle principal (cámara + YOLO + audio)

### Controles de teclado:
| Tecla | Acción |
|-------|--------|
| `Q` / `Esc` | Salir |
| `V` | Activar/desactivar voz |
| `M` | Alternar modelo (propio ↔ COCO) |
| `R` | Resetear cooldowns de alertas |

> **Nota:** Si ves la ventana congelada, haz clic sobre ella antes de presionar teclas.

In [16]:
import cv2
import sys
import time
from ultralytics import YOLO


def cargar_modelo(path, device):
    """Carga un modelo YOLO y lo mueve al dispositivo."""
    print(f"[INFO] Cargando modelo: {path}")
    m = YOLO(path)
    m.to(device)
    print(f"[INFO] Modelo cargado en {device.upper()} ✅")
    return m


def bucle_principal():
    """
    Bucle principal de detección con cámara, YOLO y alertas de voz.
    """
    # ── Estado mutable dentro del bucle ──
    estado = {
        "uso_modelo_propio": USE_MODELO_PROPIO,
        "voz_activa":        True,
    }

    dispositivo = verificar_gpu()

    # Cargar ambos modelos al inicio para que el cambio en tiempo real
    # sea instantáneo (sin recargar desde disco)
    print("\n[INFO] Precargando modelos...")
    modelos = {}
    try:
        modelos["propio"] = cargar_modelo(RUTA_MODELO_PROPIO, dispositivo)
    except Exception as e:
        print(f"[WARN] No se pudo cargar modelo propio: {e}")
        print("       Usando solo modelo COCO.")
        modelos["propio"] = None
        estado["uso_modelo_propio"] = False

    modelos["coco"] = cargar_modelo(RUTA_MODELO_COCO, dispositivo)

    # ── Abrir cámara ──
    print(f"\n[INFO] Abriendo cámara {CAMARA_INDEX}...")
    cap = cv2.VideoCapture(CAMARA_INDEX)
    #cap = cv2.VideoCapture("http://10.131.8.17:4747/video")
    if not cap.isOpened():
        print(f"[ERROR] No se pudo abrir cámara {CAMARA_INDEX}.")
        print("        Prueba cambiando CAMARA_INDEX en la Celda 2.")
        return

    cap.set(cv2.CAP_PROP_FRAME_WIDTH,  1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

    print("\n" + "=" * 55)
    print("  Visión Asistida — Sistema de Audio Espacial v2.0")
    print("=" * 55)
    print("  [Q/Esc] Salir   [V] Voz   [M] Modelo   [R] Reset")
    print("=" * 55 + "\n")

    hablar("Sistema listo. Iniciando detección.")

    tiempo_ant = time.time()
    fps = 0.0

    while True:
        ret, frame = cap.read()
        if not ret:
            print("[ERROR] No se pudo leer frame de la cámara.")
            break

        h_f, w_f = frame.shape[:2]
        area_frame = h_f * w_f

        # ── Seleccionar modelo y clases activos ──
        if estado["uso_modelo_propio"] and modelos["propio"] is not None:
            modelo_activo  = modelos["propio"]
            clases_activas = CLASES_MODELO_PROPIO
            filtrar        = False
        else:
            modelo_activo  = modelos["coco"]
            clases_activas = CLASES_MODELO_COCO
            filtrar        = True

        # ── Inferencia ──
        resultados = modelo_activo(
            frame,
            conf=CONFIANZA_MIN,
            verbose=False,
            device=dispositivo,
        )

        detecciones_frame = []
        n_objetos  = 0
        n_peligros = 0

        # ── Procesar detecciones ──
        for resultado in resultados:
            for box in resultado.boxes:
                clase_id = int(box.cls[0])

                # Filtrar clases según modo
                if filtrar and clase_id not in clases_activas:
                    continue
                if not filtrar and clase_id not in clases_activas:
                    continue

                confianza    = float(box.conf[0])
                nombre_clase = clases_activas[clase_id]
                coords       = box.xyxy[0].tolist()
                x1, y1, x2, y2 = coords

                cx = int((x1 + x2) / 2)
                area_bbox = (x2 - x1) * (y2 - y1)

                zona  = calcular_zona(cx, w_f)
                nivel = calcular_nivel_alerta(area_bbox, area_frame)

                n_objetos += 1
                if nivel == "peligro":
                    n_peligros += 1

                # Guardar para motor de prioridad
                detecciones_frame.append({
                    "nombre_clase": nombre_clase,
                    "zona":         zona,
                    "nivel":        nivel,
                    "confianza":    confianza,
                })

                # Dibujar en frame
                frame = dibujar_deteccion(
                    frame, coords, nombre_clase, confianza, nivel, zona
                )

        # ── Motor de voz (solo si la voz está activa) ──
        if estado["voz_activa"] and detecciones_frame:
            evaluar_y_alertar(detecciones_frame)

        # ── FPS ──
        ahora = time.time()
        fps = 1.0 / (ahora - tiempo_ant + 1e-9)
        tiempo_ant = ahora

        # ── HUD ──
        frame = dibujar_hud(
            frame, fps, n_objetos, n_peligros,
            estado["uso_modelo_propio"], estado["voz_activa"]
        )

        cv2.imshow("Visión Asistida — Audio Espacial", frame)

        # ── Procesar teclas ──
        tecla = cv2.waitKey(1) & 0xFF

        if tecla in (ord('q'), ord('Q'), 27):       # Salir
            print("[INFO] Saliendo...")
            break

        elif tecla in (ord('v'), ord('V')):          # Toggle voz
            estado["voz_activa"] = not estado["voz_activa"]
            msg = "Voz activada" if estado["voz_activa"] else "Voz desactivada"
            print(f"[INFO] {msg}")
            if estado["voz_activa"]:
                hablar("Voz activada")

        elif tecla in (ord('m'), ord('M')):          # Toggle modelo
            if modelos["propio"] is None:
                print("[WARN] Modelo propio no disponible.")
                continue
            estado["uso_modelo_propio"] = not estado["uso_modelo_propio"]
            nombre_modo = "modelo propio" if estado["uso_modelo_propio"] else "modelo COCO"
            print(f"[INFO] Cambiado a {nombre_modo}")
            if estado["voz_activa"]:
                hablar(f"Usando {nombre_modo}")
            resetear_cooldowns()

        elif tecla in (ord('r'), ord('R')):          # Reset cooldowns
            resetear_cooldowns()
            print("[INFO] Cooldowns reseteados.")

    # ── Limpieza ──
    cap.release()
    cv2.destroyAllWindows()
    print("[INFO] Sesión finalizada.")


# ══════════════════════════════════════
#  ▶  EJECUTAR AQUÍ
# ══════════════════════════════════════
bucle_principal()

[ADVERTENCIA] GPU no disponible. Usando CPU.

[INFO] Precargando modelos...
[INFO] Cargando modelo: runs\detect\runs\detect\deteccion_cercana\weights\best.pt
[INFO] Modelo cargado en CPU ✅
[INFO] Cargando modelo: yolov8n.pt
[INFO] Modelo cargado en CPU ✅

[INFO] Abriendo cámara 0...

  Visión Asistida — Sistema de Audio Espacial v2.0
  [Q/Esc] Salir   [V] Voz   [M] Modelo   [R] Reset

[INFO] Cambiado a modelo COCO
[INFO] Cambiado a modelo propio
[INFO] Saliendo...
[INFO] Sesión finalizada.


## Celda 7 — Cierre limpio
Ejecuta esta celda si el audio quedó activo después de cerrar la ventana.

In [ ]:
import cv2
cv2.destroyAllWindows()
detener_voz()
print("[INFO] Recursos liberados correctamente.")

[INFO] Recursos liberados correctamente.


[TTS] Hilo de audio cerrado.
